In [1]:
import os

In [16]:
# Path folder utama
path = 'ug'

for root, dirs, files in os.walk(path):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        
        # Cek isi folder
        files_in_dir = os.listdir(dir_path)
        if files_in_dir:
            # Ambil nama file pertama di dalam folder
            first_file = files_in_dir[0]
            
            # Ambil nama file tanpa ekstensi
            name_without_ext = os.path.splitext(first_file)[0]
            
            # Ganti nama folder dengan nama file (tanpa spasi dan huruf kecil)
            new_dir_name = name_without_ext.replace('-', '')
            new_dir_name = new_dir_name.replace(' ', '_')
            new_dir_path = os.path.join(root, new_dir_name)
            
            # Rename folder
            os.rename(dir_path, new_dir_path)
            print(f"Renamed: {dir_path} -> {new_dir_path}")
    
    # Hanya perlu satu iterasi level folder utama
    break


Renamed: ug\Adelfia_Milka_Novtagravina___A11.2022.14637 -> ug\Adelfia_Milka_Novtagravina__A11.2022.14637
Renamed: ug\Ainur_Rahma_Miftakhul_Rizqi___A11.2022.14579 -> ug\Ainur_Rahma_Miftakhul_Rizqi__A11.2022.14579
Renamed: ug\Aisyah_Nuraini___14161 -> ug\Aisyah_Nuraini__14161
Renamed: ug\Aliffian_Ilham_Febriyana___A11.2022.14155 -> ug\Aliffian_Ilham_Febriyana__A11.2022.14155
Renamed: ug\Amalia_Putri_Dwi_Mareta___A11.2022.13977 -> ug\Amalia_Putri_Dwi_Mareta__A11.2022.13977
Renamed: ug\Bambang_Putra_Amsal_Gautama___A11.2022.13995 -> ug\Bambang_Putra_Amsal_Gautama__A11.2022.13995
Renamed: ug\Bayu_Dewanto___A11.2022.14722 -> ug\Bayu_Dewanto__A11.2022.14722
Renamed: ug\Chandra_Lukita_Buana___A11.2022.14049 -> ug\Chandra_Lukita_Buana__A11.2022.14049
Renamed: ug\Dzawil_Uqul___A11.2022.14141 -> ug\Dzawil_Uqul__A11.2022.14141
Renamed: ug\Eric_Satria_Pratama___A11.2022.14601 -> ug\Eric_Satria_Pratama__A11.2022.14601
Renamed: ug\Erna_Amalia_Widhiyanti___A11.2022.14506 -> ug\Erna_Amalia_Widhiyanti__

In [8]:
import os
import cv2
import albumentations as A
from albumentations.core.composition import OneOf
import sys

sys.path.append(os.path.abspath('../face_detection'))
from yunet import YuNet

# Inisialisasi model YuNet
model = YuNet(modelPath='../face_detection/face_detection_yunet_2023mar.onnx',
                     inputSize=[320, 320],
                     confThreshold=0.5,
                     nmsThreshold=0.3,
                     topK=5000,
                     )

# Path awal dan tujuan
input_path = 'ug'
output_path = 'augmen_face'

# Augmentasi menggunakan Albumentations
augmentation_pipeline = A.Compose([
    OneOf([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5),
        A.RandomBrightnessContrast(p=0.5)
    ], p=1.0),
    A.Resize(224, 224)  # Opsional, untuk resize
])

# Fungsi untuk menyimpan gambar augmentasi
def save_augmented_images(image, output_dir, base_filename):
    for i in range(3):  # Simpan 3 gambar augmentasi
        augmented = augmentation_pipeline(image=image)
        augmented_image = augmented['image']
        output_file = os.path.join(output_dir, f"{base_filename}_aug_{i+1}.jpg")
        cv2.imwrite(output_file, cv2.cvtColor(augmented_image, cv2.COLOR_RGB2BGR))
        print(f"Saved: {output_file}")

# Traverse folder input dan buat augmentasi
for root, dirs, files in os.walk(input_path):
    relative_path = os.path.relpath(root, input_path)
    output_dir = os.path.join(output_path, relative_path)
    
    # Buat folder tujuan jika belum ada
    os.makedirs(output_dir, exist_ok=True)
    
    for file in files:
        file_path = os.path.join(root, file)
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):  # Cek format file
            # Baca gambar
            image = cv2.imread(file_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Konversi ke RGB
            if image is None:
                print(f"Gagal membaca gambar: {file_path}")
                continue
            
            # Inference deteksi wajah
            h, w, _ = image.shape
            model.setInputSize([w, h])
            results = model.infer(image)
            
            # Jika tidak ada wajah yang terdeteksi, skip
            if results is None or len(results) == 0:
                print(f"No faces detected in {file_path}. Skipping...")
                continue
            
            # Nama dasar file tanpa ekstensi
            base_filename = os.path.splitext(file)[0].replace(' ', '_').lower()
            
            # Proses setiap wajah yang terdeteksi
            for i, result in enumerate(results):
                x, y, w, h = map(int, result[:4])  # Koordinat bounding box
                cropped_face = image[y:y+h, x:x+w]  # Crop wajah
                
                # Simpan hasil augmentasi
                save_augmented_images(cropped_face, output_dir, f"{base_filename}_face_{i+1}")


Saved: augmen_face\Adelfia_Milka_Novtagravina__A11.2022.14637\adelfia_milka_novtagravina_-_a11.2022.14637_face_1_aug_1.jpg
Saved: augmen_face\Adelfia_Milka_Novtagravina__A11.2022.14637\adelfia_milka_novtagravina_-_a11.2022.14637_face_1_aug_2.jpg
Saved: augmen_face\Adelfia_Milka_Novtagravina__A11.2022.14637\adelfia_milka_novtagravina_-_a11.2022.14637_face_1_aug_3.jpg
Saved: augmen_face\Ainur_Rahma_Miftakhul_Rizqi__A11.2022.14579\ainur_rahma_miftakhul_rizqi_-_a11.2022.14579_face_1_aug_1.jpg
Saved: augmen_face\Ainur_Rahma_Miftakhul_Rizqi__A11.2022.14579\ainur_rahma_miftakhul_rizqi_-_a11.2022.14579_face_1_aug_2.jpg
Saved: augmen_face\Ainur_Rahma_Miftakhul_Rizqi__A11.2022.14579\ainur_rahma_miftakhul_rizqi_-_a11.2022.14579_face_1_aug_3.jpg
Saved: augmen_face\Aisyah_Nuraini__14161\aisyah_nuraini_-_14161_face_1_aug_1.jpg
Saved: augmen_face\Aisyah_Nuraini__14161\aisyah_nuraini_-_14161_face_1_aug_2.jpg
Saved: augmen_face\Aisyah_Nuraini__14161\aisyah_nuraini_-_14161_face_1_aug_3.jpg
Saved: augmen